# Project 10: Character LM Tuning on WikiText-2

Compare compact char-level model configurations on a real corpus and track
reproducibility fingerprints for each run.


## Dataset Source and Download Instructions

- Dataset: WikiText-2 (Salesforce Research)
- Official source: https://blog.salesforceairesearch.com/the-wikitext-long-term-dependency-language-modeling-dataset/
- License: check official dataset terms
- Auto-download in this notebook: `datasets.load_dataset("wikitext", "wikitext-2-raw-v1")`
- Manual fallback:
  1. Download WikiText-2 from official source.
  2. Place text files under `./data/wikitext-2/`.
  3. Update loading cell to local paths.


In [ ]:
%pip -q install neurogebra torch datasets matplotlib


In [ ]:
import hashlib
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset

from neurogebra.logging.fingerprint import TrainingFingerprint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
wt = load_dataset("wikitext", "wikitext-2-raw-v1")
train_text = "\n".join(wt["train"]["text"][:2500])
val_text = "\n".join(wt["validation"]["text"][:600])

chars = sorted(set(train_text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

train_ids = torch.tensor([stoi[c] for c in train_text if c in stoi], dtype=torch.long)
val_ids = torch.tensor([stoi[c] for c in val_text if c in stoi], dtype=torch.long)
vocab_size = len(chars)

print("Train tokens:", len(train_ids))
print("Val tokens:", len(val_ids))
print("Vocab size:", vocab_size)


In [ ]:
class CharLM(nn.Module):
    def __init__(self, vocab, emb=64, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = nn.GRU(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)

    def forward(self, x, hidden=None):
        e = self.emb(x)
        out, hidden = self.rnn(e, hidden)
        logits = self.head(out)
        return logits, hidden


def make_batch(data_ids, batch_size, context_len, device):
    ix = torch.randint(0, len(data_ids) - context_len - 1, (batch_size,))
    x = torch.stack([data_ids[i:i+context_len] for i in ix]).to(device)
    y = torch.stack([data_ids[i+1:i+context_len+1] for i in ix]).to(device)
    return x, y


def run_config(name, context_len, emb_dim, hidden_dim, steps=250, batch_size=64, lr=1e-3):
    model = CharLM(vocab_size, emb=emb_dim, hidden=hidden_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    for step in range(steps):
        xb, yb = make_batch(train_ids, batch_size, context_len, device)
        opt.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward()
        opt.step()
        train_losses.append(float(loss.item()))

    model.eval()
    with torch.no_grad():
        val_batch_x, val_batch_y = make_batch(val_ids, batch_size, context_len, device)
        val_logits, _ = model(val_batch_x)
        val_loss = float(criterion(val_logits.reshape(-1, vocab_size), val_batch_y.reshape(-1)).item())

    fp = TrainingFingerprint.capture(
        model_info={"name": name, "context_len": context_len, "emb": emb_dim, "hidden": hidden_dim},
        hyperparameters={"steps": steps, "batch_size": batch_size, "lr": lr},
        dataset=hashlib.sha256(train_text.encode("utf-8")).hexdigest()[:16],
        random_seed=SEED,
    )

    return {
        "name": name,
        "context_len": context_len,
        "emb_dim": emb_dim,
        "hidden_dim": hidden_dim,
        "train_loss_last": train_losses[-1],
        "val_loss": val_loss,
        "model": model,
        "fingerprint": fp,
    }


In [ ]:
configs = [
    {"name": "cfg_small_context", "context_len": 64, "emb_dim": 64, "hidden_dim": 128},
    {"name": "cfg_large_context", "context_len": 128, "emb_dim": 96, "hidden_dim": 192},
]

results = []
for cfg in configs:
    print("Running", cfg)
    out = run_config(**cfg)
    results.append(out)
    print(f"  train_last={out['train_loss_last']:.4f}, val={out['val_loss']:.4f}")
    print(out["fingerprint"].format_text())

best = min(results, key=lambda r: r["val_loss"])
print("Best config:", best["name"], "val_loss=", best["val_loss"])

out_dir = Path("./logs_charlm_tuning")
out_dir.mkdir(parents=True, exist_ok=True)
for r in results:
    fp_path = out_dir / f"{r['name']}_fingerprint.json"
    with fp_path.open("w", encoding="utf-8") as f:
        json.dump(r["fingerprint"].to_dict(), f, indent=2)


In [ ]:
names = [r["name"] for r in results]
val_losses = [r["val_loss"] for r in results]

plt.figure(figsize=(7, 4))
plt.bar(names, val_losses)
plt.ylabel("Validation loss")
plt.title("CharLM Config Comparison on WikiText-2")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Simple generation from best model
model = best["model"]
model.eval()
start = "The "
context_len = best["context_len"]
context = torch.tensor([stoi[c] for c in start if c in stoi], dtype=torch.long, device=device).unsqueeze(0)
generated = list(start)
hidden = None

with torch.no_grad():
    for _ in range(300):
        logits, hidden = model(context[:, -context_len:], hidden)
        probs = torch.softmax(logits[:, -1, :], dim=-1)
        idx = torch.multinomial(probs, num_samples=1)
        ch = itos[int(idx.item())]
        generated.append(ch)
        context = torch.cat([context, idx], dim=1)

print("".join(generated))
